In [ ]:
import subprocess
import os

path_2015 = "/data/brussel/vo/000/bvo00033/vsc11346/Exposure/2.Data_WP/2015/"
path_2030 = "/data/brussel/vo/000/bvo00033/vsc11346/Exposure/2.Data_WP/2030/"
path_out  = "/data/brussel/vo/000/bvo00033/vsc11346/Exposure/4.Processing_outputs/"
os.makedirs(path_out, exist_ok=True)

for year, path_in in [("2015", path_2015), ("2030", path_2030)]:

    archivos = sorted([
        f"{path_in}{f}" for f in os.listdir(path_in)
        if "_T_" in f and f.endswith(".tif")
    ])

    print(f"\n{'='*55}")
    print(f"Procesando {year} — {len(archivos)} países")
    print(f"{'='*55}")

    mosaico_100m = f"{path_out}worldpop_under18_{year}_SA_100m.tif"
    archivos_str = " ".join(archivos)

    # PASO 1: Mosaico de todos los países a 100m con gdalwarp
    cmd_merge = f"""
    source /etc/profile.d/modules.sh
    module --force purge
    module load GDAL/3.10.0-foss-2024a
    gdalwarp \
        -srcnodata -99999 \
        -dstnodata -99999 \
        -ot Float32 \
        -co COMPRESS=LZW \
        -co TILED=YES \
        {archivos_str} \
        {mosaico_100m}
    """

    print(f"\nPASO 1 — Creando mosaico 100m...")
    result = subprocess.run(cmd_merge, shell=True, executable="/bin/bash",
                            capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-300:])
    if result.returncode != 0:
        print(f"  ERROR: {result.stderr[:500]}")
        continue
    print(f"  ✅ Mosaico guardado: {mosaico_100m}")

print(f"\n✅ Proceso completo — archivos en {path_out}")

In [2]:
import rasterio
import numpy as np

path_out = "/data/brussel/vo/000/bvo00033/vsc11346/Exposure/Approach1/2.Processing_outputs/"

archivos = {
    "2015": path_out + "worldpop_under18_2015_SA_100m.tif",
    "2030": path_out + "worldpop_under18_2030_SA_100m.tif",
}

metadata = {}

for year, path in archivos.items():
    print(f"\n{'='*60}\n{year}\n{'='*60}")

    with rasterio.open(path) as src:
        print(f"CRS: {src.crs}")
        print(f"Dimensiones: {src.width} x {src.height} píxeles")
        print(f"Resolución: {src.res}")
        print(f"Extensión (bounds): {src.bounds}")
        print(f"Nodata declarado: {src.nodata}")
        print(f"Block size (tiling interno): {src.block_shapes}")

        # --- Estadísticas por bloques (memory-safe) ---
        n_validos = 0
        n_total = 0
        n_negativos = 0
        suma = 0.0
        vmin, vmax = np.inf, -np.inf

        for _, window in src.block_windows(1):
            block = src.read(1, window=window)
            valido = block != src.nodata
            if not valido.any():
                continue
            vals = block[valido]
            n_validos += vals.size
            n_total += block.size
            suma += vals.sum(dtype=np.float64)  # float64 para evitar overflow/precisión
            n_negativos += (vals < 0).sum()
            vmin = min(vmin, vals.min())
            vmax = max(vmax, vals.max())

        print(f"\nPíxeles válidos: {n_validos:,} de {n_total:,} ({100*n_validos/n_total:.1f}%)")
        print(f"Min: {vmin:.4f}  |  Max: {vmax:.4f}")
        print(f"Suma total (proxy de población <18 en el raster): {suma:,.0f}")
        print(f"Píxeles negativos (fuera de nodata): {n_negativos}")

        metadata[year] = {"bounds": src.bounds, "res": src.res, "shape": (src.height, src.width)}

print(f"\n{'='*60}\nConsistencia 2015 vs 2030\n{'='*60}")
print(f"Mismos bounds: {metadata['2015']['bounds'] == metadata['2030']['bounds']}")
print(f"Misma resolución: {metadata['2015']['res'] == metadata['2030']['res']}")
print(f"Misma forma (shape): {metadata['2015']['shape'] == metadata['2030']['shape']}")


2015
CRS: EPSG:4326
Dimensiones: 96173 x 86788 píxeles
Resolución: (0.00083333333, 0.00083333333)
Extensión (bounds): BoundingBox(left=-109.45416694885, bottom=-56.525832771230014, right=-29.31000060276, top=15.797500272809986)
Nodata declarado: -99999.0
Block size (tiling interno): [(256, 256)]

Píxeles válidos: 119,225,094 de 2,128,281,600 (5.6%)
Min: 0.0000  |  Max: 1808.0062
Suma total (proxy de población <18 en el raster): 120,944,199
Píxeles negativos (fuera de nodata): 0

2030
CRS: EPSG:4326
Dimensiones: 96173 x 86788 píxeles
Resolución: (0.00083333333, 0.00083333333)
Extensión (bounds): BoundingBox(left=-109.45416694885, bottom=-56.525832771230014, right=-29.31000060276, top=15.797500272809986)
Nodata declarado: -99999.0
Block size (tiling interno): [(256, 256)]

Píxeles válidos: 121,780,722 de 2,128,216,064 (5.7%)
Min: 0.0000  |  Max: 1612.4232
Suma total (proxy de población <18 en el raster): 105,915,797
Píxeles negativos (fuera de nodata): 0

Consistencia 2015 vs 2030
Mismo